- Pages (Check)
- Number of deaths (Check)
    - ERROR
- Clean of fields
- QC detainee $ ocrurrences 
- Compare to 500 

In [2]:
import polars as pl
import altair as alt
from pathlib import Path
import json

In [4]:
root = Path('..')
parquet_path = root / 'data/jails-data/final files/output/jails_person_records.parquet'
complete_df_path = root / 'data/jails-data/final files/output/jails_pdfs_full.parquet'
df = pl.read_parquet(parquet_path)
df_complete = pl.read_parquet(complete_df_path)
df.head()

Report ID,Name,DOB,Date_Confined,Arresting_Charge
str,str,str,str,str
"""Mateos, Gustavo 1-14_p1""",null,null,null,"""posse es"""
"""Mateos, Gustavo 1-14_p1""","""_ Mateos, Gustavo""","""1/22/1995""",null,"""""01/24/2020"""
"""Searle, Robert 1-21_p1""",null,null,null,"""| METHAMPHETAMINE"""
"""Searle, Robert 1-21_p1""","""| Searle, Robert |""",null,null,""": 11/14/1981"""
"""Sutton, Micah 1-6_p1""",null,null,"""01/06/2021""","""98/08/1983"""


In [13]:
type(df_complete['Table Contents'])

polars.series.series.Series

In [18]:
type(df_complete[3500]['Table Contents'])

polars.series.series.Series

In [14]:
type(df_complete[2993]['Table Contents'])

polars.series.series.Series

In [24]:
print(df_complete[3500]['Table Contents'][0].to_list())

['Antonio D Hall', '9/20/1991', '1/10/2024', 'Burglary']


In [12]:
print(df_complete[2993]['Table Contents'])

shape: (1,)
Series: 'Table Contents' [list[str]]
[
	["Ordonez, Robert F.", "08/12/1981", … "Crim.Damage Govt.Property"]
]


In [5]:
df_cleaned = df.with_columns(
    pl.col("Name")
    .str.replace_all(r"[;|_\-!~\"“”]", "")  # Remove ; | _ - ! ~ " “ ”
    .str.strip_chars()
    .alias("Cleaned Name")
)

In [6]:
df_cleaned

Report ID,Name,DOB,Date_Confined,Arresting_Charge,Cleaned Name
str,str,str,str,str,str
"""Mateos, Gustavo 1-14_p1""",null,null,null,"""posse es""",null
"""Mateos, Gustavo 1-14_p1""","""_ Mateos, Gustavo""","""1/22/1995""",null,"""""01/24/2020""","""Mateos, Gustavo"""
"""Searle, Robert 1-21_p1""",null,null,null,"""| METHAMPHETAMINE""",null
"""Searle, Robert 1-21_p1""","""| Searle, Robert |""",null,null,""": 11/14/1981""","""Searle, Robert"""
"""Sutton, Micah 1-6_p1""",null,null,"""01/06/2021""","""98/08/1983""",null
…,…,…,…,…,…
"""UO - FOIA January 2023_p608""",null,null,null,"""we Prey, x¢""",null
"""UO - FOIA January 2023_p608""","""‘Aiyders""",null,null,"""‘t (OABG991""""","""‘Aiyders"""
"""UO - FOIA January 2023_p609""","""Beasley, Shiloe""","""06/24/1994""","""08/01/2022""","""Agg Batt/Corr Emp/Cnt Blood,""","""Beasley, Shiloe"""


In [7]:
df_complete.columns

['Deceased Name',
 'Deceased Cause',
 'Deceased Date and Time',
 'Deceased on Suicide Watch',
 'Deceased Reporter',
 'Deceased Examined by Physician',
 'Deceased Signs of Illness',
 'Facility Name',
 'Facility Type',
 'RD Number',
 'Phone Number',
 'Address',
 'Date',
 'Time of Day',
 'AM or PM',
 'Occurrence',
 'Table Contents',
 'Injuries?',
 'Resulting Death?',
 'process',
 'Report ID',
 'OCR_Confidence',
 'OCR_Word_Count',
 'OCR_Text_Detected',
 'Cook County?']

In [7]:
# merge with complet df to have the whol table of contents per pdf
df_merged = df_cleaned.join(
    df_complete.select(['Report ID', 'Table Contents']),
    on='Report ID',
    how='left'
)


In [ ]:
df_merged

In [ ]:
df_cleaned = df.with_columns(
    pl.col("Name")
    .str.replace_all(r"[;|_\-!~\"“”]", "")  # Remove ; | _ - ! ~ " “ ”
    .str.strip_chars()
    .alias("Cleaned Name")
)

name_counts = (
    df_cleaned
    .filter(
        pl.col("Cleaned Name").is_not_null()
    )
    .group_by("Cleaned Name")
    .len()
    .sort("len", descending=True)
    .head(50)
    .to_pandas()
)

# Bar chart of top 50 names with custom color
alt.Chart(name_counts).mark_bar(color='#E0AC3BA6').encode(
    x=alt.X('len:Q', title='Frequency'),
    y=alt.Y('Cleaned Name:N', sort='-x', title='Name'),
    tooltip=['Cleaned Name', 'len']
).properties(
    width=600,
    height=800,
    title='Top 50 Most Frequent Names'
)

alt.Chart(...)

# This file seems to have a different format

Lets open the data we are sure to have


In [4]:
#And all the database:
root = Path('..')
parquet_path = root / 'data/jails-data/output/jails_pdfs.parquet'
df = pl.read_parquet(parquet_path)
list(df['Facility Name'][0:10])


['-acility Name: McHenry County Adult Correctional Facility\n',
 "‘acility Name: Kane County Sheriff's Office\n",
 'Fax: (217) 558-4004\n‘acility Name: \\ tei So ( ae ( She cA Ofce\n',
 '‘acility Name: /¥lATTHSOns = otice = bn FAPAMEAT\nTT eee Py ml a ei fF gt Mm Ahh sn\n',
 '-acility Name: Lake County Adult Corrections\n',
 '‘acility Name: Williamson County Jail\n',
 '-acility Name: Lake County Jail\n',
 'Facility Name: Lake County Jail\n',
 'Fax: (217) 522-3906\n-acility Name: McLean County Jail\n',
 '‘acility Name: Macoupin County Sheriff\n']

In [5]:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ')  # Remove \n, \\, \/
        .str.replace_all(r'\n', ' ')  # Remove actual newlines
        .str.replace_all(r'[\\\/]+', ' ')  # Clean slashes
        .str.replace_all(r'\s+', ' ')  # Multiple spaces to single
        .str.strip_chars()  # Trim whitespace
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        .alias('Cleaned Facility Name')
    )

# Apply the cleaning
df_cleaned = clean_facility_name(df)
df_cleaned['Facility Name', 'Cleaned Facility Name']

Facility Name,Cleaned Facility Name
str,str
"""-acility Name: McHenry County …","""McHenry County Adult Correctio…"
"""‘acility Name: Kane County She…","""Kane County Sheriff's Office"""
"""Fax: (217) 558-4004 ‘acility N…","""tei So ae She cA Ofce"""
"""‘acility Name: /¥lATTHSOns = o…","""lATTHSOns otice bn FAPAMEAT TT…"
"""-acility Name: Lake County Adu…","""Lake County Adult Corrections"""
…,…
"""SANs (att) Vee VI ——es- ‘acili…",null
"""Facility Name: Lake County She…","""Lake County Sheriff's Office"""
"""-acility Name: Lake County Jai…","""Lake County Jail"""


In [7]:
#Cleanning the date
list(df_cleaned['Date'][:10])

['Date of Occurrence: 01/09/2021\n',
 'Date of Occurrence: 01/27/2021\n',
 'aAqaaress. eet o~ SF Never\nPate af Praacemmamane f—) 7 + ed } fs 9 ng\n',
 'Date of Occurrence: i/ (Z]2 ( Tir\n',
 'Date of Occurrence: 01/15/2021\n',
 'Date of Occurrence: 01/14/2021\n',
 'Date of Occurrence: 01/19/2021\n',
 'Date of Occurrence: 01/13/2021\n',
 'AGaress. 14 West STORE SUCCL\nStreet\nRiwten af Manairrramnna: (111/12£/9N%1 ~\n',
 'Date of Occurrence: 01/26/2021 T\n']

In [10]:
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

df_cleaned = clean_date_occurrence(df_cleaned)
df_cleaned.select(["Date","Cleaned Date"]).head(10)

Date,Cleaned Date
str,str
"""Date of Occurrence: 01/09/2021…","""01/09/2021"""
"""Date of Occurrence: 01/27/2021…","""01/27/2021"""
"""aAqaaress. eet o~ SF Never Pat…",null
"""Date of Occurrence: i/ (Z]2 ( …",null
"""Date of Occurrence: 01/15/2021…","""01/15/2021"""
"""Date of Occurrence: 01/14/2021…","""01/14/2021"""
"""Date of Occurrence: 01/19/2021…","""01/19/2021"""
"""Date of Occurrence: 01/13/2021…","""01/13/2021"""
"""AGaress. 14 West STORE SUCCL S…",null


In [18]:
excel_path = root / 'data/jails-data/Initial 500 restraint chairs database.xlsx'
df_ex = pl.read_excel(excel_path, sheet_name='Chairs 19-22')
#df_ex.head()


Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 20, falling back to string


In [17]:
df_ex.filter(pl.col("PDF name").str.contains("Binder"))

PDF folder,PDF name,Date received,R.D. Number,Facility name,Date of occurrence,Time of occurrence,Type of occurrence 1,Type of occurrence 2,Type of occurrence 3,Other (specify),Name first,Name last,DOB,Date confined,Arresting charge 1,Arresting charge 2,Injuries?,Death?,Reported by,Notes
str,str,date,str,str,date,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2021_06""","""Binder1.pdf""",2021-06-14,null,"""Clinton County Sheriff's Offic…",2021-06-01,1899-12-31 20:43:00,"""Restraints used""",null,null,"""Chair""","""Taylor""","""Ross""","""1997-12-17 00:00:00""","""2021-06-01 00:00:00""","""Poss Cont Sub""","""Clinton County Warrant""","""No""","""No""",null,null
"""2021_06""","""Binder1.pdf""",2021-06-22,null,"""Dekalb County Sheriff's Office""",2021-06-16,1899-12-31 12:55:00,"""Restraints used""",null,null,"""Chair""","""Jarrett M.""","""Presswood""","""1999-06-05 00:00:00""","""2021-06-11 00:00:00""","""Agg Batt to PO, Resist PO, lea…",null,"""No""","""No""",null,null
"""2021_06""","""Binder1.pdf""",2021-06-15,null,"""Kane County Sheriff's Office""",2021-06-09,1899-12-31 04:15:00,"""Restraints used""",null,null,"""Chair""","""Biannca Angelina""","""Bestler""","""1993-10-09 00:00:00""","""2021-06-09 00:00:00""","""DWLS, DWLS, UPC""",null,"""No""",null,null,null
"""2021_06""","""Binder1.pdf""",2021-06-22,null,"""Kane County Sheriff's Office""",2021-06-17,1899-12-31 18:15:00,"""Restraints used""",null,null,"""Chair""","""Michael J.""","""Corbett""","""1958-02-23 00:00:00""","""2021-06-17 00:00:00""","""Criminal Tresspass (x6)""",null,"""Corbett-right arm scrape""",null,null,null
"""2021_06""","""Binder1.pdf""",2021-06-23,null,"""Kane County Sheriff's Office""",2021-06-21,1899-12-31 08:22:00,"""Restraints used""","""OC spray used""",null,"""Chair""","""James Michael""","""Morgan""","""1967-08-21 00:00:00""","""2021-04-13 00:00:00""","""Agg.Battery(x3), UPSMV""",null,"""No""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2021_06""","""Binder1.pdf""",2021-06-28,null,"""Peoria County Jail""",2021-06-27,1899-12-31 20:05:00,"""Restraints used""",null,null,"""Chair""","""Callie""","""Nash""","""1967-06-22 00:00:00""","""2021-06-27 00:00:00""","""AGG BATT TO NURSE X4""",null,"""No""","""No""",null,null
"""2021_06""","""Binder1.pdf""",2021-07-02,null,"""St. Clair County Sheriff's Dep…",2021-06-30,1899-12-31 19:20:00,"""Restraints used""",null,null,"""Chair""","""Kiarra""","""Daniels""","""1995-03-07 00:00:00""","""2021-04-28 00:00:00""","""Damage to State Property""",null,"""No""","""No""",null,null
"""2021_06""","""Binder1.pdf""",2021-06-23,null,"""St. Clair County Sheriff's Dep…",2021-06-18,1899-12-31 16:30:00,"""Restraints used""",null,null,"""Chair""","""Steven""","""Goestenkors""","""1979-04-03 00:00:00""","""2021-01-28 00:00:00""","""Agg Batt""",null,"""Head wound""","""No""",null,null


In [19]:
df_cleaned

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Type,Facility Name,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?,Cleaned Facility Name,Cleaned Date
str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,f64,i64,bool,str,str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""-acility Name: McHenry County …","""__ Telephone #: 815-338-4741 _…","""Address: 2200 N. Seminary Ave.…","""Date of Occurrence: 01/09/2021…","""“ wy wiew Time of Occurrence: …","""p.m. """,[],"[""vee ence"", ""Webster, Tiffany"", … ""- Domestic Battery/ Agg Batt""]","""No [] Yes (brieflydescribe) = …","""No ""","""Webster, Tiffany 1-9_p1""",87.506757,296,true,null,"""McHenry County Adult Correctio…","""01/09/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""‘acility Name: Kane County She…","""__ Telephone #: 630-232-6840 ""","""daress: 3/W/)) IL Koute 56 Sui…","""Date of Occurrence: 01/27/2021…","""_ Time of Occurrence: 204: ""","""No filled box found""","[""fhe nied iets eat "", ""Suicide Attempt (method) Drinking fo: ""]","[""Cross, Anton Nelson"", ""| 02/04/1999"", … ""Murder 1*, Agg Battery""]","""No L[] Yes, (briefly describe)…","""No filled box found""","""Cross, Anton 1-27 (suicide att…",87.860068,293,true,null,"""Kane County Sheriff's Office""","""01/27/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""- ""","""Fax: (217) 558-4004 ‘acility N…","""__ Telephone #: ( b1 16 OLY Ba…","""‘acility Name: SASL a! eae A C…","""aAqaaress. eet o~ SF Never Pat…","""EAE AOL EE Ve a —_—_ City Stat…","""Error creating ROI: zero-lengt…","[""oo. .. Deleant Si. "", "". . g. @ pe. Ce | om | "", … ""ault among Detainees . ""]","[""Horm [Po tv ce)"", ""0S"" 0- em ol AT 202!"", … ""_ (det""]","""—_ T™) «7 ka em. dla i eh ts ""","""iA | ""","""Paradee, Travis 1-16_p1""",81.046667,300,true,null,"""tei So ae She cA Ofce""",null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""ant. Inc ""","""‘acility Name: /¥lATTHSOns = o…","""elephone #: _70$ -St3~ 3) 3c _…","""a LT LO CC CC LT tn PE oe Oe ~…","""Date of Occurrence: i/ (Z]2 ( …","""ime of Occurrence: _O4 Se phan…","""Error: less than two boxes fou…","[""ault among Detaine "", ""aulton Staff , "", ""ape Ch Escape Attempt | ""]","[""WELIIIC CS WIVOlVEU"", ""wae eee"", … ""Ae bel pIvVAriie""]","""Yes, attach coroner's report o…","""Error: less than two boxes fou…","""Sampson, Kevin 1-12 (suicide a…",78.514851,303,true,null,"""lATTHSOns otice bn FAPAMEAT TT…",null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""-acility Name: Lake County Adu…","""_. Telephone #: 847 377-4099 ""","""Address: 29 South County Stree…","""Date of Occurrence: 01/15/2021…","""_ Time of Occurrence: 2:50 ""","""pm, """,[],"[""UGLGAMICCS UivVivauw"", ""-10n8/ 1984"", … ""“Hut, Anthony L160833""]","""Yes, (briefly describe): Inmat…","""No ""","""Huff, Anthony 1-15_p1""",88.664516,310,true,null,"""Lake County Adult Corrections""","""01/15/2021"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""SANs (att) Vee VI ——es- ‘acili…","""per: Telenhone #: 3()9-634-445…","""""","""Address: 301 NMAXWELL RD Stree…","""PEORIA IL City Stat ""","""Error: less than two boxes fou…",[],"[""(1 Fighting among Detainees"", ""C1] Restraints Used"", … ""| TAZ CO HOLD""]","""No""","""Error: less than two boxes fou…","""UO - FOIA January 2023_p611""",86.477612,268,true,null,null,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""">| ""","""Facility Name: Lake County She…","""___ Telephone #: (847) 377-409…","""Address: 29S. MLK Jr. Ave. Wau…","""Date of Occurrence: 01/31/2023…","""_ Time of Occurrence: 4:50 ""","""a.m. """,[],"[""WOLAHItes mivoliveu"", ""ae ae"", … ""oo spammers""]","""No""","

In [20]:
# Analysis with the excel Finding Binder pdf
df_cleaned.filter(pl.col("Report ID").str.contains("Binder"))

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Type,Facility Name,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?,Cleaned Facility Name,Cleaned Date
str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,f64,i64,bool,str,str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""Facility Name: Lake County Jai…","""__ Telephone #: 847-377-4099 ""","""Address: 29 S. County Street W…","""Date of Occurrence: 06/30/2021…","""Time of Occurrence: 9:00 ""","""p.m. """,[],"[""_ 0702 1 992"", ""“Alarcon, Alfredo"", … ""“Age Battery (Great Bodily Harm""]","""No [J Yes, (briefly describe):…","""No ""","""Binder1_p1""",88.487889,289,true,null,"""Lake County Jail""","""06/30/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Error: less than two boxes fou…","""‘acility Name: Cook County Dep…","""Telephone #: 773.674.3474 e) W…","""\ddress: Z2/UU o. CAIITOPNId V…","""Date of Occurrence: 6/9/2021 T…","""HHMe OF VCCUTTCTICE, 1.IU.UYU …","""Check time, no AM or PM for Co…","[""Fighting amonc ""]","[""Dakari J Allen"", ""8/24/1999"", … ""8/22/2020""]","""No""","""No filled box found""","""Binder1_p2""",87.804965,282,true,"""Cook County""","""Cook County Department Of Corr…","""06/09/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Error: less than two boxes fou…","""‘acility Name: Cook County Dep…","""Telephone #: 773.674.3474 O IL…","""Aqaress: SIUM Y4- SGM MIiVioiV…","""Date of Occurrence: 6/29/2021 …","""SSS WE Ow EE Eee Nes ew T1 Ghi…","""Check time, no AM or PM for Co…","[""Fighting amonc ""]","[""VCldINees IVvVOIVeag"", ""Totianna Allen"", … ""12/5/2020""]","""No""","""No filled box found""","""Binder1_p3""",87.717314,283,true,"""Cook County""","""Cook County Department Of Corr…","""06/29/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""‘acility Name: Iroquois County…","""_ Telephone #: APE ""","""\daress: 7 a ae a aw wae ww ew…","""Date of Occurrence: 06/23/2024…","""_ Time of Occurrence: — 10:12 ""","""Error creating ROI: zero-lengt…","[""Suicide Attempt (method) bed st ""]","[""Harold E. Alliss."", ""~ 02/10/2984 06/17/2021 Poss Meth, viol OP""]","""Yes. (briefly describe) NO nec…","""No ""","""Binder1_p4""",87.551613,310,true,null,"""Iroquois County Jail""","""06/23/2024"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Chicagc ""","""address: 727 E. 11tthStreet_ C…","""0 60628 —tiy”~*«CSStat@s Zip C…","""Date of Occurrence: June 1 oF …","""Type of Occurrence: [) Suicide…","""Suicide Attempt (method) shirt…","""Error: less than two boxes fou…",[],"[""| A onzo, Ramon"", ""|| May 1 1 992 June 15, 2021 | Warrant & Simple Assau""]","""No [J] Yes, attach coroner's r…","""Error creating ROI: zero-lengt…","""Binder1_p5""",86.688136,295,true,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""Facility Name: Madison County …","""__._ Telephone #: 618-692-1065…","""Address: 405 Randle Street Edw…","""Date of Occurrence: 06/04/2021…","""bate A wow _ Time of Occurrenc…","""p.m. ""","[""Restraints Used "", ""3attery CJ Riot ""]","[""William Tyler Woods"", ""08/21/1986"", … ""Homicide/Aggravated Arson""]","""No [|] Yes, (briefly describe)…","""No ""","""Binder1_p427""",87.643599,289,true,null,"""Madison County Sheriff's Offic…","""06/04/2021"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""Facility Name: Madison County …","""____ Telephone #: 618-692-1065…","""Address: 405 Randle Street Edw…","""Date of Occurrence: 06/15/2021…","""tet A wan _ Time of Occurrence…","""p.m. ""","[""=ighting among De "", ""3attery CJ Rio ""]","[""Cody Lee James Wright"", ""09/21/1994"", … ""Burglary""]","""Yes, (briefly describe): | Tim…","""No ""","""Binder1_p428"""